# Essai avec ce modèle qui n'a pas été sélectionné 
## Modèle sélectionné --> voir lightGBM.ipynb

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.xgboost
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    average_precision_score,
    f1_score,
    recall_score,
    precision_score,
)
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

In [3]:
## chargement des données 
df = pd.read_csv('data/train_engineered.csv')

X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

print('Shape X :', X.shape)
print('Distribution TARGET :\n', y.value_counts())

Shape X : (307511, 396)
Distribution TARGET :
 TARGET
0.0    282686
1.0     24825
Name: count, dtype: int64


In [4]:
## nettoyage des infinis (produits par les agregations dans kernel2)
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
print('Infinis restants :', np.isinf(X.select_dtypes('number').values).sum())

Infinis restants : 0


In [5]:
## train/test
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train :', X_train.shape)
print('Val   :', X_val.shape)

Train : (246008, 396)
Val   : (61503, 396)


In [ ]:
## On paramètre ML Flow pour notre usage :
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('home-credit-scoring')

mlflow.xgboost.autolog(
    log_models=False,
    log_input_examples=False,
    log_model_signatures=False
)

## Modèle 1

In [ ]:
def cout_metier(y_true, y_pred, cout_fn=10, cout_fp=1):
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return cout_fn * fn + cout_fp * fp

# XGBoost gère le déséquilibre via scale_pos_weight 
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

params = {
    'n_estimators'    : 300,
    'max_depth'       : 5,
    'learning_rate'   : 0.05,
    'scale_pos_weight': scale_pos_weight,
    'random_state'    : 42,
    'n_jobs'          : -1,
    'eval_metric'     : 'auc',
}

with mlflow.start_run(run_name='xgboost_scale_pos_weight',
                      tags={'type': 'scale_pos_weight', 'seuil': 'optimise_cout_metier'},
                      description='XGBoost avec scale_pos_weight pour gérer le déséquilibre, seuil optimisé sur coût métier FN=10.FP'):

    model_xgb = XGBClassifier(**params)
    model_xgb.fit(X_train, y_train)

    # validation croisée
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model_xgb, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    mlflow.log_metric('cv_auc_mean', cv_scores.mean())
    mlflow.log_metric('cv_auc_std',  cv_scores.std())

    # probabilités sur val
    y_pred_proba = model_xgb.predict_proba(X_val)[:, 1]
    mlflow.log_metric('auc',           roc_auc_score(y_val, y_pred_proba))
    mlflow.log_metric('avg_precision', average_precision_score(y_val, y_pred_proba))

    # seuil optimal basé sur le coût métier
    fpr, tpr, thresholds = roc_curve(y_val, y_pred_proba)
    couts = []
    for threshold in thresholds:
        y_pred_t = (y_pred_proba > threshold).astype(int)
        couts.append(cout_metier(y_val, y_pred_t))
    optimal_threshold = thresholds[np.argmin(couts)]
    mlflow.log_param('seuil_optimal', round(float(optimal_threshold), 3))

    # métriques au seuil optimal
    y_pred = (y_pred_proba > optimal_threshold).astype(int)
    mlflow.log_metric('recall_minority',    recall_score(y_val, y_pred, pos_label=1))
    mlflow.log_metric('precision_minority', precision_score(y_val, y_pred, pos_label=1))
    mlflow.log_metric('f1',                f1_score(y_val, y_pred, pos_label=1))

    # coût métier
    cout      = cout_metier(y_val, y_pred)
    cout_naif = cout_metier(y_val, np.zeros(len(y_val)).astype(int))
    ratio_vs_naif = cout / cout_naif
    mlflow.log_metric('cout_metier',   cout)
    mlflow.log_metric('ratio_vs_naif', ratio_vs_naif)

    print(f'CV AUC        : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print(f'AUC val       : {roc_auc_score(y_val, y_pred_proba):.4f}')
    print(f'Seuil optimal : {optimal_threshold:.3f}')
    print(f'Recall        : {recall_score(y_val, y_pred, pos_label=1):.3f}')
    print(f'Précision     : {precision_score(y_val, y_pred, pos_label=1):.3f}')
    print(f'Ratio vs naïf : {ratio_vs_naif:.3f}')
    print(f'Le modèle coûte {ratio_vs_naif:.0%} du coût d\'un modèle qui ne détecte rien.')

# Le meilleur pour le moment. 

CV AUC        : 0.7740 ± 0.0011
AUC val       : 0.7812
Seuil optimal : 0.510
Recall        : 0.677
Précision     : 0.193
Ratio vs naïf : 0.606
Le modèle coûte 61% du coût d'un modèle qui ne détecte rien.
